# 02 — Features

Builds team-week EPA stats with rolling windows, then assembles the game-level feature table.

Key principle: **all features are shifted by 1 game** — a team's stats for week N use only data through week N-1, so we never leak the future into the past.

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/raw')
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pbp = pl.read_parquet(DATA_DIR / 'pbp.parquet')
schedules = pl.read_parquet(DATA_DIR / 'schedules.parquet')
print('PBP:', pbp.shape, '| Schedules:', schedules.shape)

PBP: (1279628, 372) | Schedules: (7276, 46)


## Team-week EPA stats (offense + defense)

In [2]:
# Only real offensive plays — drop kneels, spikes, special teams, penalties-only
plays = pbp.filter(
    pl.col('play_type').is_in(['pass', 'run']) &
    pl.col('epa').is_not_null() &
    (pl.col('season_type') == 'REG')
)

off_stats = (
    plays.group_by(['season', 'week', 'posteam'])
    .agg([
        pl.col('epa').mean().alias('off_epa_play'),
        pl.col('success').mean().alias('off_success_rate'),
        (pl.col('epa') > 0.5).mean().alias('off_explosive_rate'),
        pl.col('epa').count().alias('off_plays'),
    ])
    .rename({'posteam': 'team'})
)

def_stats = (
    plays.group_by(['season', 'week', 'defteam'])
    .agg([
        pl.col('epa').mean().alias('def_epa_play'),
        pl.col('success').mean().alias('def_success_rate'),
        (pl.col('epa') > 0.5).mean().alias('def_explosive_rate'),
    ])
    .rename({'defteam': 'team'})
)

team_week = off_stats.join(def_stats, on=['season','week','team'], how='inner').to_pandas()
team_week = team_week.sort_values(['team','season','week']).reset_index(drop=True)
print('Team-week rows:', len(team_week))
team_week.head()

Team-week rows: 13928


,season,week,team,off_epa_play,off_success_rate,off_explosive_rate,off_plays,def_epa_play,def_success_rate,def_explosive_rate
0,1999,1,ARI,-0.133430,0.345238,0.226190,84,-0.273381,0.363636,0.181818
1,1999,2,ARI,-0.427857,0.263158,0.140351,57,-0.250607,0.369231,0.200000
2,1999,3,ARI,-0.249298,0.343750,0.281250,64,0.035640,0.428571,0.222222
3,1999,4,ARI,-0.441154,0.266667,0.150000,60,0.073966,0.418182,0.272727
4,1999,5,ARI,-0.133809,0.321429,0.232143,56,-0.224083,0.362319,0.202899


## Rolling windows (lagged by 1 — no leakage)

In [3]:
stat_cols = ['off_epa_play','off_success_rate','off_explosive_rate',
             'def_epa_play','def_success_rate','def_explosive_rate']

tw = team_week.copy()
for c in stat_cols:
    # last-4-game and last-8-game rolling avg, lagged by 1
    tw[f'{c}_l4'] = tw.groupby('team')[c].transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).mean()
    )
    tw[f'{c}_l8'] = tw.groupby('team')[c].transform(
        lambda s: s.shift(1).rolling(8, min_periods=1).mean()
    )
    # season-to-date, lagged by 1
    tw[f'{c}_ytd'] = tw.groupby(['team','season'])[c].transform(
        lambda s: s.shift(1).expanding().mean()
    )

tw.to_parquet(OUT_DIR / 'team_week.parquet')
print('Saved team_week with rolling features')
tw.filter(items=['team','season','week','off_epa_play','off_epa_play_l4','off_epa_play_ytd']).head(10)

Saved team_week with rolling features


,team,season,week,off_epa_play,off_epa_play_l4,off_epa_play_ytd
0,ARI,1999,1,-0.133430,NaN,NaN
1,ARI,1999,2,-0.427857,-0.133430,-0.133430
2,ARI,1999,3,-0.249298,-0.280643,-0.280643
3,ARI,1999,4,-0.441154,-0.270195,-0.270195
4,ARI,1999,5,-0.133809,-0.312935,-0.312935
5,ARI,1999,6,-0.272907,-0.313029,-0.277110
6,ARI,1999,8,-0.423612,-0.274292,-0.276409
7,ARI,1999,9,-0.217587,-0.317870,-0.297438
8,ARI,1999,10,-0.119031,-0.261979,-0.287457
9,ARI,1999,11,-0.128015,-0.258284,-0.268743


## QB-level rolling EPA

In [4]:
# Aggregate dropbacks per QB per game
qb_plays = pbp.filter(
    (pl.col('qb_dropback') == 1) &
    pl.col('passer_player_id').is_not_null() &
    pl.col('qb_epa').is_not_null() &
    (pl.col('season_type') == 'REG')
)
qb_week_raw = (
    qb_plays.group_by(['season','week','posteam','passer_player_id','passer_player_name'])
    .agg([
        pl.col('qb_epa').mean().alias('qb_epa_dropback'),
        pl.col('qb_epa').count().alias('qb_dropbacks'),
    ])
    .filter(pl.col('qb_dropbacks') >= 10)  # min snaps to count as starter
    .to_pandas()
    .sort_values(['passer_player_id','season','week'])
    .reset_index(drop=True)
)
print('QB-week rows (min 10 dropbacks):', len(qb_week_raw))

QB-week rows (min 10 dropbacks): 14588


In [5]:
# Rolling QB EPA — last 4 games & season-to-date, lagged by 1
qb_week_raw['qb_epa_l4'] = qb_week_raw.groupby('passer_player_id')['qb_epa_dropback'].transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean()
)
qb_week_raw['qb_epa_ytd'] = qb_week_raw.groupby(['passer_player_id','season'])['qb_epa_dropback'].transform(
    lambda s: s.shift(1).expanding().mean()
)
# Pick the team's primary QB that week = the one with most dropbacks
team_qb = (
    qb_week_raw.sort_values('qb_dropbacks', ascending=False)
    .groupby(['season','week','posteam'], as_index=False)
    .first()
    .rename(columns={'posteam':'team'})
    [['season','week','team','passer_player_id','passer_player_name','qb_epa_l4','qb_epa_ytd']]
)
# Detect QB downgrade: was THIS week's starter different from the team's season-to-date primary starter?
# Cumulative most-snapped QB per team-season (lagged)
qb_snaps = qb_week_raw.copy()
qb_snaps['cum_dropbacks'] = qb_snaps.groupby(['posteam','season','passer_player_id'])['qb_dropbacks'].cumsum().shift(1).fillna(0)
primary = (
    qb_snaps.sort_values('cum_dropbacks', ascending=False)
    .groupby(['season','week','posteam'], as_index=False)
    .first()[['season','week','posteam','passer_player_id']]
    .rename(columns={'posteam':'team','passer_player_id':'primary_qb_id'})
)
team_qb = team_qb.merge(primary, on=['season','week','team'], how='left')
team_qb['qb_downgrade'] = (team_qb['passer_player_id'] != team_qb['primary_qb_id']).astype(int)
team_qb.to_parquet(OUT_DIR / 'team_qb.parquet')
print(f'Saved team_qb: {len(team_qb)} team-weeks')
team_qb.head(10)

Saved team_qb: 13920 team-weeks


,season,week,team,passer_player_id,passer_player_name,qb_epa_l4,qb_epa_ytd,primary_qb_id,qb_downgrade
0,1999,1,ARI,00-0013042,J.Plummer,NaN,NaN,00-0013042,0
1,1999,1,ATL,00-0002876,C.Chandler,NaN,NaN,00-0002876,0
2,1999,1,BUF,00-0005363,D.Flutie,NaN,NaN,00-0005363,0
3,1999,1,CAR,00-0001218,S.Beuerlein,NaN,NaN,00-0001218,0
4,1999,1,CHI,00-0010560,S.Matthews,NaN,NaN,00-0010560,0
5,1999,1,CIN,00-0001335,J.Blake,NaN,NaN,00-0001335,0
6,1999,1,CLE,00-0004230,T.Detmer,NaN,NaN,00-0004230,0
7,1999,1,DAL,00-0000104,T.Aikman,NaN,NaN,00-0000104,0
8,1999,1,DEN,00-0006423,B.Griese,NaN,NaN,00-0006423,0
9,1999,1,DET,00-0000865,C.Batch,NaN,NaN,00-0000865,0


## Game-level feature table

In [6]:
sched = schedules.to_pandas()
games = sched[(sched['game_type'] == 'REG') & sched['result'].notna()].copy()
# nflverse convention: result = home_score - away_score (positive = home win)
games['margin'] = games['home_score'] - games['away_score']
print('Completed regular-season games:', len(games))

Completed regular-season games: 6967


In [7]:
# Build separate home/away feature tables and merge
feat_cols = [c for c in tw.columns if c not in ['season','week','team']]

home_feats = tw.rename(columns={'team':'home_team', **{c: f'home_{c}' for c in feat_cols}})
away_feats = tw.rename(columns={'team':'away_team', **{c: f'away_{c}' for c in feat_cols}})

g = games.merge(home_feats, on=['season','week','home_team'], how='left')
g = g.merge(away_feats, on=['season','week','away_team'], how='left')

# Differential features (home - away) — these are usually the strongest signal
for c in stat_cols:
    for w in ['l4','l8','ytd']:
        g[f'diff_{c}_{w}'] = g[f'home_{c}_{w}'] - g[f'away_{c}_{w}']

g['rest_diff'] = g['home_rest'] - g['away_rest']

# Merge QB features (team_qb has season/week/team + qb_epa_l4/ytd + qb_downgrade)
tqb = pd.read_parquet(OUT_DIR / 'team_qb.parquet')
h_qb = tqb.rename(columns={'team':'home_team','qb_epa_l4':'home_qb_epa_l4','qb_epa_ytd':'home_qb_epa_ytd','qb_downgrade':'home_qb_downgrade'})[['season','week','home_team','home_qb_epa_l4','home_qb_epa_ytd','home_qb_downgrade']]
a_qb = tqb.rename(columns={'team':'away_team','qb_epa_l4':'away_qb_epa_l4','qb_epa_ytd':'away_qb_epa_ytd','qb_downgrade':'away_qb_downgrade'})[['season','week','away_team','away_qb_epa_l4','away_qb_epa_ytd','away_qb_downgrade']]
g = g.merge(h_qb, on=['season','week','home_team'], how='left')
g = g.merge(a_qb, on=['season','week','away_team'], how='left')
# Fill missing QB stats (rookies, missing data) with league-avg dropback EPA ~ 0.05
for c in ['home_qb_epa_l4','home_qb_epa_ytd','away_qb_epa_l4','away_qb_epa_ytd']:
    g[c] = g[c].fillna(0.05)
for c in ['home_qb_downgrade','away_qb_downgrade']:
    g[c] = g[c].fillna(0).astype(int)
# QB diff features
g['diff_qb_epa_l4']  = g['home_qb_epa_l4']  - g['away_qb_epa_l4']
g['diff_qb_epa_ytd'] = g['home_qb_epa_ytd'] - g['away_qb_epa_ytd']
g['diff_qb_downgrade'] = g['home_qb_downgrade'] - g['away_qb_downgrade']  # +1 home downgraded, -1 away downgraded

g.to_parquet(OUT_DIR / 'game_features.parquet')
print(f'Saved {len(g)} games with features')
g[['game_id','season','week','home_team','away_team','margin','spread_line','diff_qb_epa_l4','diff_qb_downgrade']].head(10)

Saved 6967 games with features


,game_id,season,week,home_team,away_team,margin,spread_line,diff_qb_epa_l4,diff_qb_downgrade
0,1999_01_MIN_ATL,1999,1,ATL,MIN,-3,-4.0,0.0,0
1,1999_01_KC_CHI,1999,1,CHI,KC,3,-3.0,0.0,0
2,1999_01_PIT_CLE,1999,1,CLE,PIT,-43,-6.0,0.0,0
3,1999_01_OAK_GB,1999,1,GB,OAK,4,9.0,0.0,0
4,1999_01_BUF_IND,1999,1,IND,BUF,17,-3.0,0.0,0
5,1999_01_SF_JAX,1999,1,JAX,SF,38,5.5,0.0,0
6,1999_01_CAR_NO,1999,1,NO,CAR,9,3.5,0.0,0
7,1999_01_NE_NYJ,1999,1,NYJ,NE,-2,7.0,0.0,1
8,1999_01_ARI_PHI,1999,1,PHI,ARI,-1,-3.0,0.0,0
9,1999_01_DET_SEA,1999,1,SEA,DET,-8,9.5,0.0,0


## Done
Next: open `03_train_backtest.ipynb` to fit Elo + XGBoost and run the walk-forward backtest.